# 问题二：投票结合方式分析

## 目标
利用问题一估算的观众投票数，分析两种投票结合方式（排名制 vs 百分比制）的差异。

## 核心任务
1. 将两种方式应用于所有34季，比较淘汰结果差异
2. 分析哪种方式更偏向观众投票
3. 分析争议选手在两种方式下的表现
4. 分析评委决胜规则的影响
5. 给出未来采用方式的推荐

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 设置
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')

FIGSIZE_NORMAL = (10, 6)
FIGSIZE_WIDE = (12, 6)
COLORS = {
    'primary': '#4682B4',
    'secondary': '#FF7F50',
    'accent': '#228B22',
    'neutral': '#708090'
}

np.random.seed(42)

In [ ]:
# 读取问题一的投票估算结果
vote_estimates = pd.read_csv('../问题一/vote_estimates.csv')
season_info = pd.read_csv('../数据预处理/season_summary.csv')
df_processed = pd.read_csv('../数据预处理/data_processed.csv')

print(f'投票估算数据: {len(vote_estimates)} 条')
print(f'赛季数: {vote_estimates["season"].nunique()}')
vote_estimates.head()

## 第一部分：两种投票方式的实现

### 1.1 方式说明

**排名制 (Rank-based)**：S1-2, S28-34
- 计算评委得分排名 $R_{judge}$ 和观众投票排名 $R_{fan}$
- 综合排名 $R_{total} = R_{judge} + R_{fan}$
- 综合排名**最高者**淘汰

**百分比制 (Percentage-based)**：S3-27
- 计算评委得分百分比 $P_{judge} = J_i / \sum J$
- 综合百分比 $P_{total} = P_{judge} + P_{fan}$
- 综合百分比**最低者**淘汰

In [ ]:
def apply_rank_method(scores, votes):
    """
    排名制：返回每位选手的综合排名（排名越高越差）
    """
    n = len(scores)
    # 评委排名：得分越高排名越靠前（数值越小）
    judge_ranks = stats.rankdata(-np.array(scores), method='min')
    # 观众排名：投票越高排名越靠前
    fan_ranks = stats.rankdata(-np.array(votes), method='min')
    # 综合排名
    total_ranks = judge_ranks + fan_ranks
    return total_ranks, judge_ranks, fan_ranks

def apply_percentage_method(scores, votes):
    """
    百分比制：返回每位选手的综合百分比（百分比越低越差）
    """
    scores = np.array(scores)
    votes = np.array(votes)
    # 评委百分比
    judge_pct = scores / scores.sum()
    # 观众百分比（投票比例本身就是百分比）
    fan_pct = votes / votes.sum() if votes.sum() > 0 else votes
    # 综合百分比
    total_pct = judge_pct + fan_pct
    return total_pct, judge_pct, fan_pct

def get_eliminated_by_method(scores, votes, method):
    """
    根据指定方式返回被淘汰者的索引
    """
    if method == 'rank':
        total_ranks, _, _ = apply_rank_method(scores, votes)
        return np.argmax(total_ranks)  # 排名最高（数值最大）者淘汰
    else:
        total_pct, _, _ = apply_percentage_method(scores, votes)
        return np.argmin(total_pct)  # 百分比最低者淘汰

def get_bottom_two_by_method(scores, votes, method):
    """
    返回排名最后两位的索引（用于评委决胜规则）
    """
    if method == 'rank':
        total_ranks, _, _ = apply_rank_method(scores, votes)
        sorted_indices = np.argsort(total_ranks)[::-1]  # 从高到低
    else:
        total_pct, _, _ = apply_percentage_method(scores, votes)
        sorted_indices = np.argsort(total_pct)  # 从低到高
    
    return sorted_indices[:2]  # 返回最后两位

## 第二部分：将两种方式应用于所有赛季

In [ ]:
def analyze_week(week_data):
    """
    分析某周在两种方式下的淘汰结果
    """
    if len(week_data) < 2:
        return None
    
    contestants = week_data['contestant'].tolist()
    scores = week_data['total_score'].values
    votes = week_data['estimated_vote_prop'].values
    statuses = week_data['status'].tolist()
    
    # 实际淘汰者
    actual_eliminated = [c for c, s in zip(contestants, statuses) if s == 'eliminated_this_week']
    
    if len(actual_eliminated) == 0:
        return None  # 无淘汰周跳过
    
    # 排名制下的淘汰
    rank_elim_idx = get_eliminated_by_method(scores, votes, 'rank')
    rank_eliminated = contestants[rank_elim_idx]
    
    # 百分比制下的淘汰
    pct_elim_idx = get_eliminated_by_method(scores, votes, 'percentage')
    pct_eliminated = contestants[pct_elim_idx]
    
    # 排名制下的最后两位（用于评委决胜）
    rank_bottom_two_idx = get_bottom_two_by_method(scores, votes, 'rank')
    rank_bottom_two = [contestants[i] for i in rank_bottom_two_idx]
    
    # 百分比制下的最后两位
    pct_bottom_two_idx = get_bottom_two_by_method(scores, votes, 'percentage')
    pct_bottom_two = [contestants[i] for i in pct_bottom_two_idx]
    
    # 计算得分和投票的排名
    judge_ranks = stats.rankdata(-scores, method='min')
    vote_ranks = stats.rankdata(-votes, method='min')
    
    # 排名制淘汰者的评委和投票排名
    rank_elim_judge_rank = judge_ranks[rank_elim_idx]
    rank_elim_vote_rank = vote_ranks[rank_elim_idx]
    
    # 百分比制淘汰者的排名
    pct_elim_judge_rank = judge_ranks[pct_elim_idx]
    pct_elim_vote_rank = vote_ranks[pct_elim_idx]
    
    return {
        'season': week_data['season'].iloc[0],
        'week': week_data['week'].iloc[0],
        'n_contestants': len(contestants),
        'actual_method': week_data['voting_method'].iloc[0],
        'actual_eliminated': actual_eliminated[0] if len(actual_eliminated) == 1 else str(actual_eliminated),
        'rank_eliminated': rank_eliminated,
        'pct_eliminated': pct_eliminated,
        'methods_agree': rank_eliminated == pct_eliminated,
        'rank_bottom_two': str(rank_bottom_two),
        'pct_bottom_two': str(pct_bottom_two),
        'rank_elim_judge_rank': rank_elim_judge_rank,
        'rank_elim_vote_rank': rank_elim_vote_rank,
        'pct_elim_judge_rank': pct_elim_judge_rank,
        'pct_elim_vote_rank': pct_elim_vote_rank
    }

# 对所有周进行分析
comparison_results = []

for (season, week), group in vote_estimates.groupby(['season', 'week']):
    # 只分析有淘汰的周（得分>0的选手）
    active_data = group[group['total_score'] > 0]
    result = analyze_week(active_data)
    if result is not None:
        comparison_results.append(result)

comparison_df = pd.DataFrame(comparison_results)
print(f'分析周数: {len(comparison_df)}')
comparison_df.head(10)

In [ ]:
# 统计两种方式的一致性
agreement_rate = comparison_df['methods_agree'].mean()
disagreement_count = (~comparison_df['methods_agree']).sum()

print('='*60)
print('【两种方式一致性分析】')
print('='*60)
print(f'两种方式一致率: {agreement_rate:.1%}')
print(f'不一致周数: {disagreement_count}/{len(comparison_df)}')

In [ ]:
# 查看不一致的周
disagreements = comparison_df[~comparison_df['methods_agree']].copy()
print(f'\n不一致的周 ({len(disagreements)} 周):')
print(disagreements[['season', 'week', 'actual_method', 'rank_eliminated', 'pct_eliminated', 'actual_eliminated']].to_string())

## 第三部分：哪种方式更偏向观众投票？

分析方法：比较两种方式下淘汰者的「评委排名」与「投票排名」。
- 如果某方式淘汰的选手「评委排名更差、投票排名更好」→ 该方式更偏向评委
- 如果某方式淘汰的选手「投票排名更差、评委排名更好」→ 该方式更偏向观众

In [ ]:
# 分析淘汰者的排名特征
print('='*60)
print('【淘汰者排名特征分析】')
print('='*60)

# 排名制淘汰者的平均排名
rank_method_stats = {
    'avg_judge_rank': comparison_df['rank_elim_judge_rank'].mean(),
    'avg_vote_rank': comparison_df['rank_elim_vote_rank'].mean()
}

# 百分比制淘汰者的平均排名
pct_method_stats = {
    'avg_judge_rank': comparison_df['pct_elim_judge_rank'].mean(),
    'avg_vote_rank': comparison_df['pct_elim_vote_rank'].mean()
}

print(f'\n排名制淘汰者:')
print(f'  平均评委排名: {rank_method_stats["avg_judge_rank"]:.2f}')
print(f'  平均投票排名: {rank_method_stats["avg_vote_rank"]:.2f}')

print(f'\n百分比制淘汰者:')
print(f'  平均评委排名: {pct_method_stats["avg_judge_rank"]:.2f}')
print(f'  平均投票排名: {pct_method_stats["avg_vote_rank"]:.2f}')

# 偏向性判断
rank_judge_bias = rank_method_stats['avg_judge_rank'] - rank_method_stats['avg_vote_rank']
pct_judge_bias = pct_method_stats['avg_judge_rank'] - pct_method_stats['avg_vote_rank']

print(f'\n偏向性分析（正值=更偏向评委，淘汰者评委排名更差）:')
print(f'  排名制偏向: {rank_judge_bias:.2f}')
print(f'  百分比制偏向: {pct_judge_bias:.2f}')

In [ ]:
# 计算更精细的偏向性指标
def calculate_method_bias(comparison_df):
    """
    计算每种方式对评委/观众的偏向程度
    偏向指标 = (淘汰者评委排名 - 淘汰者投票排名) / 选手数
    正值 = 更偏向评委（淘汰者评委排名更差）
    负值 = 更偏向观众（淘汰者投票排名更差）
    """
    results = []
    
    for _, row in comparison_df.iterrows():
        n = row['n_contestants']
        
        # 排名制偏向
        rank_bias = (row['rank_elim_judge_rank'] - row['rank_elim_vote_rank']) / n
        
        # 百分比制偏向
        pct_bias = (row['pct_elim_judge_rank'] - row['pct_elim_vote_rank']) / n
        
        results.append({
            'season': row['season'],
            'week': row['week'],
            'rank_bias': rank_bias,
            'pct_bias': pct_bias,
            'bias_diff': rank_bias - pct_bias  # 正值=排名制更偏向评委
        })
    
    return pd.DataFrame(results)

bias_df = calculate_method_bias(comparison_df)

print('='*60)
print('【方法偏向性指标】')
print('='*60)
print(f'排名制平均偏向: {bias_df["rank_bias"].mean():.4f} (正=偏向评委)')
print(f'百分比制平均偏向: {bias_df["pct_bias"].mean():.4f}')
print(f'\n差异 (排名制-百分比制): {bias_df["bias_diff"].mean():.4f}')

if bias_df['bias_diff'].mean() > 0:
    print('结论: 排名制相对更偏向评委（淘汰评委排名更差的选手）')
else:
    print('结论: 百分比制相对更偏向评委')

In [ ]:
# 可视化：两种方式的偏向性比较
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 图1：偏向性分布对比
ax1 = axes[0]
ax1.hist(bias_df['rank_bias'], bins=20, alpha=0.6, label='Rank-based', color=COLORS['primary'])
ax1.hist(bias_df['pct_bias'], bins=20, alpha=0.6, label='Percentage-based', color=COLORS['secondary'])
ax1.axvline(0, color='black', linestyle='--', linewidth=1)
ax1.axvline(bias_df['rank_bias'].mean(), color=COLORS['primary'], linestyle='-', linewidth=2)
ax1.axvline(bias_df['pct_bias'].mean(), color=COLORS['secondary'], linestyle='-', linewidth=2)
ax1.set_xlabel('Bias Score (positive = favors judges)')
ax1.set_ylabel('Frequency')
ax1.legend()

# 图2：两种方式一致率按赛季
ax2 = axes[1]
agreement_by_season = comparison_df.groupby('season')['methods_agree'].mean()
ax2.bar(agreement_by_season.index, agreement_by_season.values, color=COLORS['primary'])
ax2.axhline(agreement_rate, color='red', linestyle='--', label=f'Overall: {agreement_rate:.1%}')
ax2.set_xlabel('Season')
ax2.set_ylabel('Agreement Rate')
ax2.set_ylim(0, 1.1)
ax2.legend()

plt.tight_layout()
plt.savefig('figures/fig1_method_comparison.pdf', bbox_inches='tight')
plt.show()

print('='*60)
print('【图1数据特征】')
print(f'   排名制平均偏向: {bias_df["rank_bias"].mean():.4f}')
print(f'   百分比制平均偏向: {bias_df["pct_bias"].mean():.4f}')
print(f'   两种方式一致率: {agreement_rate:.1%}')
print('='*60)

## 第四部分：争议选手分析

分析4位争议选手在两种方式下的情况：
1. Jerry Rice (S2)
2. Billy Ray Cyrus (S4)
3. Bristol Palin (S11)
4. Bobby Bones (S27)

In [ ]:
# 争议选手信息
controversial_contestants = [
    ('Jerry Rice', 2, '5周评委得分最低，获亚军'),
    ('Billy Ray Cyrus', 4, '6周评委得分最低，获第5名'),
    ('Bristol Palin', 11, '12次评委得分最低，获季军'),
    ('Bobby Bones', 27, '评委得分持续偏低，夺冠')
]

def analyze_controversial_contestant(name, season, vote_estimates, comparison_df):
    """
    分析争议选手在两种方式下的情况
    """
    # 获取该选手参与的所有周
    contestant_weeks = vote_estimates[(vote_estimates['contestant'] == name) & 
                                       (vote_estimates['season'] == season) &
                                       (vote_estimates['total_score'] > 0)]
    
    if len(contestant_weeks) == 0:
        return None
    
    # 统计在两种方式下该选手被淘汰/进入bottom 2的次数
    weeks_survived = len(contestant_weeks)
    
    # 获取该赛季的比较数据
    season_comparisons = comparison_df[comparison_df['season'] == season]
    
    # 统计该选手在每周的位置
    rank_would_eliminate = 0
    pct_would_eliminate = 0
    rank_bottom_two_count = 0
    pct_bottom_two_count = 0
    
    for _, row in season_comparisons.iterrows():
        if row['rank_eliminated'] == name:
            rank_would_eliminate += 1
        if row['pct_eliminated'] == name:
            pct_would_eliminate += 1
        if name in row['rank_bottom_two']:
            rank_bottom_two_count += 1
        if name in row['pct_bottom_two']:
            pct_bottom_two_count += 1
    
    # 计算平均排名
    avg_judge_rank = []
    avg_vote_rank = []
    
    for week in contestant_weeks['week'].unique():
        week_all = vote_estimates[(vote_estimates['season'] == season) & 
                                   (vote_estimates['week'] == week) &
                                   (vote_estimates['total_score'] > 0)]
        if len(week_all) == 0:
            continue
        
        scores = week_all['total_score'].values
        votes = week_all['estimated_vote_prop'].values
        contestants = week_all['contestant'].tolist()
        
        if name not in contestants:
            continue
        
        idx = contestants.index(name)
        judge_ranks = stats.rankdata(-scores, method='min')
        vote_ranks = stats.rankdata(-votes, method='min')
        
        avg_judge_rank.append(judge_ranks[idx])
        avg_vote_rank.append(vote_ranks[idx])
    
    return {
        'contestant': name,
        'season': season,
        'weeks_survived': weeks_survived,
        'rank_would_eliminate': rank_would_eliminate,
        'pct_would_eliminate': pct_would_eliminate,
        'rank_bottom_two_count': rank_bottom_two_count,
        'pct_bottom_two_count': pct_bottom_two_count,
        'avg_judge_rank': np.mean(avg_judge_rank) if avg_judge_rank else None,
        'avg_vote_rank': np.mean(avg_vote_rank) if avg_vote_rank else None
    }

# 分析所有争议选手
controversy_analysis = []
for name, season, note in controversial_contestants:
    result = analyze_controversial_contestant(name, season, vote_estimates, comparison_df)
    if result:
        result['note'] = note
        controversy_analysis.append(result)

controversy_df = pd.DataFrame(controversy_analysis)
print('争议选手两种方式分析:')
controversy_df

In [ ]:
# 详细输出
print('='*60)
print('【争议选手详细分析】')
print('='*60)

for _, row in controversy_df.iterrows():
    print(f"\n{row['contestant']} (Season {row['season']})")
    print(f"  背景: {row['note']}")
    print(f"  参赛周数: {row['weeks_survived']}")
    print(f"  平均评委排名: {row['avg_judge_rank']:.1f}")
    print(f"  平均投票排名: {row['avg_vote_rank']:.1f}")
    print(f"  排名制下被淘汰周数: {row['rank_would_eliminate']}")
    print(f"  百分比制下被淘汰周数: {row['pct_would_eliminate']}")
    print(f"  排名制下进入Bottom 2周数: {row['rank_bottom_two_count']}")
    print(f"  百分比制下进入Bottom 2周数: {row['pct_bottom_two_count']}")
    
    # 判断哪种方式对该选手更有利
    if row['rank_would_eliminate'] > row['pct_would_eliminate']:
        print(f"  → 百分比制对该选手更有利")
    elif row['rank_would_eliminate'] < row['pct_would_eliminate']:
        print(f"  → 排名制对该选手更有利")
    else:
        print(f"  → 两种方式结果相同")

In [ ]:
# 可视化：争议选手在两种方式下的被淘汰次数
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(controversy_df))
width = 0.35

bars1 = ax.bar(x - width/2, controversy_df['rank_would_eliminate'], width,
               label='Rank-based (would eliminate)', color=COLORS['primary'])
bars2 = ax.bar(x + width/2, controversy_df['pct_would_eliminate'], width,
               label='Percentage-based (would eliminate)', color=COLORS['secondary'])

ax.set_xlabel('Controversial Contestants')
ax.set_ylabel('Weeks Would Be Eliminated')
ax.set_xticks(x)
ax.set_xticklabels([f"{row['contestant']}\n(S{row['season']})" 
                    for _, row in controversy_df.iterrows()])
ax.legend()

plt.tight_layout()
plt.savefig('figures/fig2_controversial_methods.pdf', bbox_inches='tight')
plt.show()

print('='*60)
print('【图2数据特征】')
for _, row in controversy_df.iterrows():
    print(f"   {row['contestant']}: 排名制淘汰{row['rank_would_eliminate']}次, 百分比制淘汰{row['pct_would_eliminate']}次")
print('='*60)

## 第五部分：评委决胜规则分析

从S28开始，节目采用「评委从排名最后两位中选择淘汰对象」的规则。

分析这一规则的影响：如果评委倾向于淘汰评委得分更低的选手，会改变多少淘汰结果？

In [ ]:
def simulate_judge_tiebreaker(week_data, method='rank'):
    """
    模拟评委决胜规则：评委从Bottom 2中选择评委得分更低的选手淘汰
    返回：(原始淘汰者, 评委决胜后淘汰者, 是否改变)
    """
    if len(week_data) < 2:
        return None, None, None
    
    contestants = week_data['contestant'].tolist()
    scores = week_data['total_score'].values
    votes = week_data['estimated_vote_prop'].values
    
    # 原始淘汰者
    original_elim_idx = get_eliminated_by_method(scores, votes, method)
    original_elim = contestants[original_elim_idx]
    
    # 获取Bottom 2
    bottom_two_idx = get_bottom_two_by_method(scores, votes, method)
    
    # 评委从Bottom 2中选择评委得分更低的
    bottom_two_scores = [scores[i] for i in bottom_two_idx]
    judge_choice_idx = bottom_two_idx[np.argmin(bottom_two_scores)]
    judge_choice = contestants[judge_choice_idx]
    
    changed = original_elim != judge_choice
    
    return original_elim, judge_choice, changed

# 对所有周模拟评委决胜规则
tiebreaker_results = []

for (season, week), group in vote_estimates.groupby(['season', 'week']):
    active_data = group[group['total_score'] > 0]
    if len(active_data) < 2:
        continue
    
    # 实际淘汰者
    statuses = active_data['status'].tolist()
    contestants = active_data['contestant'].tolist()
    actual_eliminated = [c for c, s in zip(contestants, statuses) if s == 'eliminated_this_week']
    
    if len(actual_eliminated) == 0:
        continue
    
    # 排名制下的评委决胜
    rank_orig, rank_judge, rank_changed = simulate_judge_tiebreaker(active_data, 'rank')
    # 百分比制下的评委决胜
    pct_orig, pct_judge, pct_changed = simulate_judge_tiebreaker(active_data, 'percentage')
    
    if rank_orig is None:
        continue
    
    tiebreaker_results.append({
        'season': season,
        'week': week,
        'actual_eliminated': actual_eliminated[0] if len(actual_eliminated) == 1 else str(actual_eliminated),
        'rank_original': rank_orig,
        'rank_with_tiebreaker': rank_judge,
        'rank_changed': rank_changed,
        'pct_original': pct_orig,
        'pct_with_tiebreaker': pct_judge,
        'pct_changed': pct_changed
    })

tiebreaker_df = pd.DataFrame(tiebreaker_results)
print(f'评委决胜规则分析: {len(tiebreaker_df)} 周')

In [ ]:
# 统计评委决胜规则的影响
rank_change_rate = tiebreaker_df['rank_changed'].mean()
pct_change_rate = tiebreaker_df['pct_changed'].mean()

print('='*60)
print('【评委决胜规则影响分析】')
print('='*60)
print(f'\n排名制 + 评委决胜:')
print(f'  结果改变率: {rank_change_rate:.1%} ({tiebreaker_df["rank_changed"].sum()}/{len(tiebreaker_df)}周)')

print(f'\n百分比制 + 评委决胜:')
print(f'  结果改变率: {pct_change_rate:.1%} ({tiebreaker_df["pct_changed"].sum()}/{len(tiebreaker_df)}周)')

print(f'\n结论: 评委决胜规则在排名制下改变{rank_change_rate:.1%}的结果，在百分比制下改变{pct_change_rate:.1%}的结果')

In [ ]:
# 分析评委决胜规则对争议选手的影响
print('='*60)
print('【评委决胜规则对争议选手的影响】')
print('='*60)

for name, season, note in controversial_contestants:
    season_tiebreaker = tiebreaker_df[tiebreaker_df['season'] == season]
    
    # 在排名制+评委决胜下，该选手会被淘汰多少次
    rank_tiebreaker_elim = (season_tiebreaker['rank_with_tiebreaker'] == name).sum()
    rank_orig_elim = (season_tiebreaker['rank_original'] == name).sum()
    
    # 在百分比制+评委决胜下
    pct_tiebreaker_elim = (season_tiebreaker['pct_with_tiebreaker'] == name).sum()
    pct_orig_elim = (season_tiebreaker['pct_original'] == name).sum()
    
    print(f"\n{name} (S{season}):")
    print(f"  排名制: 原始淘汰{rank_orig_elim}次 → 评委决胜后{rank_tiebreaker_elim}次")
    print(f"  百分比制: 原始淘汰{pct_orig_elim}次 → 评委决胜后{pct_tiebreaker_elim}次")
    
    if rank_tiebreaker_elim > rank_orig_elim:
        print(f"  → 排名制+评委决胜对该选手更不利")
    elif pct_tiebreaker_elim > pct_orig_elim:
        print(f"  → 百分比制+评委决胜对该选手更不利")

In [ ]:
# 可视化：评委决胜规则的影响
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 图1：按赛季的结果改变率
ax1 = axes[0]
change_by_season_rank = tiebreaker_df.groupby('season')['rank_changed'].mean()
change_by_season_pct = tiebreaker_df.groupby('season')['pct_changed'].mean()

x = np.arange(len(change_by_season_rank))
width = 0.35
ax1.bar(x - width/2, change_by_season_rank.values, width, label='Rank-based', color=COLORS['primary'], alpha=0.7)
ax1.bar(x + width/2, change_by_season_pct.values, width, label='Percentage-based', color=COLORS['secondary'], alpha=0.7)
ax1.set_xlabel('Season')
ax1.set_ylabel('Change Rate with Judge Tiebreaker')
ax1.set_xticks(x[::5])
ax1.set_xticklabels(change_by_season_rank.index[::5])
ax1.legend()

# 图2：总体改变率对比
ax2 = axes[1]
methods = ['Rank\nOriginal', 'Rank +\nTiebreaker', 'Pct\nOriginal', 'Pct +\nTiebreaker']
# 计算与实际结果的一致率
rank_orig_correct = (tiebreaker_df['rank_original'] == tiebreaker_df['actual_eliminated']).mean()
rank_tb_correct = (tiebreaker_df['rank_with_tiebreaker'] == tiebreaker_df['actual_eliminated']).mean()
pct_orig_correct = (tiebreaker_df['pct_original'] == tiebreaker_df['actual_eliminated']).mean()
pct_tb_correct = (tiebreaker_df['pct_with_tiebreaker'] == tiebreaker_df['actual_eliminated']).mean()

values = [rank_orig_correct, rank_tb_correct, pct_orig_correct, pct_tb_correct]
colors_list = [COLORS['primary'], COLORS['primary'], COLORS['secondary'], COLORS['secondary']]
alphas = [0.6, 1.0, 0.6, 1.0]

bars = ax2.bar(methods, values, color=colors_list)
for bar, alpha in zip(bars, alphas):
    bar.set_alpha(alpha)
ax2.set_ylabel('Consistency with Actual Elimination')
ax2.set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('figures/fig3_tiebreaker_analysis.pdf', bbox_inches='tight')
plt.show()

print('='*60)
print('【图3数据特征】')
print(f'   排名制原始一致率: {rank_orig_correct:.1%}')
print(f'   排名制+评委决胜一致率: {rank_tb_correct:.1%}')
print(f'   百分比制原始一致率: {pct_orig_correct:.1%}')
print(f'   百分比制+评委决胜一致率: {pct_tb_correct:.1%}')
print('='*60)

## 第六部分：推荐分析

In [ ]:
# 综合分析与推荐
print('='*70)
print('【综合分析与推荐】')
print('='*70)

print('\n1. 两种方式的差异:')
print(f'   - 一致率: {agreement_rate:.1%}')
print(f'   - 不一致周数: {disagreement_count}')

print('\n2. 偏向性分析:')
print(f'   - 排名制偏向: {bias_df["rank_bias"].mean():.4f}')
print(f'   - 百分比制偏向: {bias_df["pct_bias"].mean():.4f}')
if bias_df['rank_bias'].mean() > bias_df['pct_bias'].mean():
    print('   - 结论: 排名制更偏向评委意见')
else:
    print('   - 结论: 百分比制更偏向评委意见')

print('\n3. 争议选手分析:')
for _, row in controversy_df.iterrows():
    diff = row['pct_would_eliminate'] - row['rank_would_eliminate']
    if diff > 0:
        print(f"   - {row['contestant']}: 排名制更有利 (差{abs(diff)}次)")
    elif diff < 0:
        print(f"   - {row['contestant']}: 百分比制更有利 (差{abs(diff)}次)")
    else:
        print(f"   - {row['contestant']}: 两种方式相同")

print('\n4. 评委决胜规则影响:')
print(f'   - 排名制下改变率: {rank_change_rate:.1%}')
print(f'   - 百分比制下改变率: {pct_change_rate:.1%}')

print('\n' + '='*70)
print('【推荐】')
print('='*70)

# 基于分析给出推荐
print('\n基于以上分析，我们的推荐如下：')
print()

# 推荐方式
if pct_orig_correct > rank_orig_correct:
    recommended_method = 'percentage'
    print('推荐方式: 百分比制 (Percentage-based)')
    print('理由: 与实际淘汰结果一致性更高，更能反映综合表现')
else:
    recommended_method = 'rank'
    print('推荐方式: 排名制 (Rank-based)')
    print('理由: 与实际淘汰结果一致性更高')

print()
# 是否推荐评委决胜
if rank_tb_correct > rank_orig_correct or pct_tb_correct > pct_orig_correct:
    print('推荐采用评委决胜规则: 是')
    print('理由: 评委决胜规则可以提高结果的一致性，减少争议')
else:
    print('推荐采用评委决胜规则: 否')
    print('理由: 评委决胜规则对结果一致性影响不大')

## 第七部分：结果汇总与保存

In [ ]:
# 保存结果
comparison_df.to_csv('method_comparison.csv', index=False)
controversy_df.to_csv('controversial_analysis.csv', index=False)
tiebreaker_df.to_csv('tiebreaker_analysis.csv', index=False)
bias_df.to_csv('bias_analysis.csv', index=False)

print('结果文件已保存:')
print('  - method_comparison.csv')
print('  - controversial_analysis.csv')
print('  - tiebreaker_analysis.csv')
print('  - bias_analysis.csv')

In [ ]:
# ============================================================
# 建模结果汇总（供论文引用）
# ============================================================

print('\n' + '='*70)
print('【问题二建模结果汇总】')
print('='*70)

print('\n1. 两种方式比较')
print(f'   分析周数: {len(comparison_df)}')
print(f'   两种方式一致率: {agreement_rate:.1%}')
print(f'   不一致周数: {disagreement_count}')

print('\n2. 偏向性指标')
print(f'   排名制平均偏向: {bias_df["rank_bias"].mean():.4f}')
print(f'   百分比制平均偏向: {bias_df["pct_bias"].mean():.4f}')

print('\n3. 与实际结果一致率')
print(f'   排名制: {rank_orig_correct:.1%}')
print(f'   百分比制: {pct_orig_correct:.1%}')
print(f'   排名制+评委决胜: {rank_tb_correct:.1%}')
print(f'   百分比制+评委决胜: {pct_tb_correct:.1%}')

print('\n4. 评委决胜规则影响')
print(f'   排名制下改变率: {rank_change_rate:.1%}')
print(f'   百分比制下改变率: {pct_change_rate:.1%}')

print('\n5. 争议选手总结')
for _, row in controversy_df.iterrows():
    print(f"   {row['contestant']} (S{row['season']}): 排名制淘汰{row['rank_would_eliminate']}次, 百分比制淘汰{row['pct_would_eliminate']}次")

print('\n6. 生成的图片')
figures = [
    'fig1_method_comparison.pdf',
    'fig2_controversial_methods.pdf',
    'fig3_tiebreaker_analysis.pdf'
]
for i, fig_name in enumerate(figures, 1):
    print(f'   图{i}: {fig_name}')

print('\n' + '='*70)
print('以上数值可直接用于论文撰写')
print('='*70)

# 保存汇总
results_summary = {
    'comparison_weeks': len(comparison_df),
    'agreement_rate': agreement_rate,
    'disagreement_count': disagreement_count,
    'rank_bias_mean': bias_df['rank_bias'].mean(),
    'pct_bias_mean': bias_df['pct_bias'].mean(),
    'rank_consistency': rank_orig_correct,
    'pct_consistency': pct_orig_correct,
    'rank_tiebreaker_consistency': rank_tb_correct,
    'pct_tiebreaker_consistency': pct_tb_correct,
    'rank_tiebreaker_change_rate': rank_change_rate,
    'pct_tiebreaker_change_rate': pct_change_rate
}

pd.DataFrame([results_summary]).to_csv('results_summary.csv', index=False)
print('\n结果汇总已保存: results_summary.csv')